In [1]:
import os
from huggingface_hub import hf_hub_download
from gliner2 import GLiNER2

In [2]:
repo_id = "fastino/gliner2-base-v1"
filenames = ["model.safetensors", "config.json", "encoder_config/config.json", "tokenizer.json", "tokenizer_config.json"]
destination = "search_models"

os.makedirs(destination, exist_ok=True)

for file in filenames:
    hf_hub_download(
        repo_id=repo_id,
        filename=file,
        local_dir=destination
    )

In [4]:
extractor = GLiNER2.from_pretrained(destination)
extractor

You are using a model of type extractor to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first


GLiNER2(
  (encoder): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128011, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-07, ele

In [5]:
main_schema = extractor.create_schema().entities({
    "author_person_name": {
        "description": (
            "Exact text span of a person's name explicitly mentioned in the query, "
            "only when it refers to an author. "
            "Extract only the name itself, exactly as written. "
            "Do not infer missing names. "
            "Do not extract descriptions, roles, generic nouns, or surrounding words."
        ),
        "dtype": "list",
        "threshold": 0.85
    },
    "book_or_series_title": {
        "description": (
            "Exact text span of a book title or series title explicitly mentioned in the query. "
            "Extract only the title itself, exactly as written. "
            "infer missing titles. "
            "Do not extract descriptive phrases, genre phrases, plot details, or surrounding words."
        ),
        "dtype": "list",
        "threshold": 0.95
    }
})

In [6]:
query = "Fantasy book with dragons in it by Brandom Mull I think"

extractor.extract(query, main_schema, threshold=0.6, include_confidence=True) 

{'entities': {'author_person_name': [{'text': 'Brandom Mull',
    'confidence': 0.9999964237213135}],
  'book_or_series_title': [{'text': 'Fantasy book with dragons in it',
    'confidence': 0.9977849721908569}]}}

In [90]:
query = "Brandon Sanderson series with a girl with long hair that takes place in Space"

schema = extractor.create_schema().entities({
    "author_person_name": {
        "description": (
            "Exact text span of a person's name explicitly mentioned in the query, "
            "only when it refers to an author. "
            "Extract only the name itself, exactly as written. "
            "Do not infer missing names. "
            "Do not extract descriptions, roles, generic nouns, or surrounding words."
        ),
        "dtype": "list",
        "threshold": 0.85
    },
    "book_or_series_title": {
        "description": (
            "Exact text span of a book title or series title explicitly mentioned in the query. "
            "Extract only the title itself, exactly as written. "
            "infer missing titles. "
            "Do not extract descriptive phrases, genre phrases, plot details, or surrounding words."
        ),
        "dtype": "list",
        "threshold": 0.2
    }
})

results = extractor.extract(query, schema, threshold=0.6, include_confidence=True)  # Default threshold
results

{'entities': {'author_person_name': [{'text': 'Brandon Sanderson',
    'confidence': 0.9993098974227905}],
  'book_or_series_title': [{'text': 'Brandon Sanderson series',
    'confidence': 0.8138813972473145}]}}

In [89]:
query = "book where there was a sword on the cover, but it looked kind of old and had oldish lettering. It had characters with powers too"

schema = extractor.create_schema().entities({
    "author_person_name": {
        "description": (
            "Exact text span of a person's name explicitly mentioned in the query, "
            "only when it refers to an author. "
            "Extract only the name itself, exactly as written. "
            "Do not infer missing names. "
            "Do not extract descriptions, roles, generic nouns, or surrounding words."
        ),
        "dtype": "list",
        "threshold": 0.85
    },
    "book_or_series_title": {
        "description": (
            "Exact text span of a book title or series title explicitly mentioned in the query. "
            "Extract only the title itself, exactly as written. "
            "infer missing titles. "
            "Do not extract descriptive phrases, genre phrases, plot details, or surrounding words."
        ),
        "dtype": "list",
        "threshold": 0.2
    }
})

results = extractor.extract(query, schema, threshold=0.6, include_confidence=True)  # Default threshold
results

{'entities': {'author_person_name': [],
  'book_or_series_title': [{'text': 'book',
    'confidence': 0.8835898637771606}]}}

In [88]:
query = "That one Defensive Baking series, I think by Kingfisher something"

schema = extractor.create_schema().entities({
    "author_person_name": {
        "description": (
            "Exact text span of a person's name explicitly mentioned in the query, "
            "only when it refers to an author. "
            "Extract only the name itself, exactly as written. "
            "Do not infer missing names. "
            "Do not extract descriptions, roles, generic nouns, or surrounding words."
        ),
        "dtype": "list",
        "threshold": 0.85
    },
    "book_or_series_title": {
        "description": (
            "Exact text span of a book title or series title explicitly mentioned in the query. "
            "Extract only the title itself, exactly as written. "
            "infer missing titles. "
            "Do not extract descriptive phrases, genre phrases, plot details, or surrounding words."
        ),
        "dtype": "list",
        "threshold": 0.2
    }
})

results = extractor.extract(query, schema, threshold=0.6, include_confidence=True)  # Default threshold
results

{'entities': {'author_person_name': [{'text': 'Kingfisher',
    'confidence': 0.9999516010284424}],
  'book_or_series_title': [{'text': 'one Defensive Baking series',
    'confidence': 0.8772921562194824}]}}

In [15]:
query = "Brandon Sanderson series with a girl with long hair that takes place in Space"

schema = extractor.create_schema().entities({
    "author": {
        "description": "The proper noun name of the author",
        "dtype": "list",
        "threshold": 0.5
    },
    "novel title": {
        "description": "The proper noun title",
        "dtype": "list",
        "threshold": 0.95
    }
})

results = extractor.extract(query, schema, threshold=0.6, include_confidence=True)  # Default threshold
results

{'entities': {'author': [{'text': 'Brandon Sanderson',
    'confidence': 0.9998915195465088}],
  'novel title': []}}

In [18]:
query = "book where there was a sword on the cover, but it looked kind of old and had oldish lettering. It had characters with powers too"

schema = extractor.create_schema().entities({
    "author": {
        "description": "The proper noun name of the author",
        "dtype": "list",
        "threshold": 0.5
    },
    "novel title": {
        "description": "The proper noun title",
        "dtype": "list",
        "threshold": 0.95
    }
})

results = extractor.extract(query, schema, threshold=0.6, include_confidence=True)  # Default threshold
results

{'entities': {'author': [], 'novel title': []}}

In [45]:
query = "That one Defensive Baking series, I think by something Kingfisher"

schema = extractor.create_schema().entities({
    "author": {
        "description": "The proper noun name of the author",
        "dtype": "list",
        "threshold": 0.5
    },
    "novel title": {
        "description": "The proper noun title",
        "dtype": "list",
        "threshold": 0.9
    }
})

results = extractor.extract(query, schema, include_confidence=True)
results

{'entities': {'author': [{'text': 'something Kingfisher',
    'confidence': 0.9999909400939941}],
  'novel title': [{'text': 'one Defensive Baking',
    'confidence': 0.9821833372116089}]}}

In [46]:
query = "Brandon Sanderson series with a girl with long hair that takes place in Space"

result = extractor.extract_entities(query, ["author", "book"],
    include_confidence=True)
result

{'entities': {'author': [{'text': 'Brandon Sanderson',
    'confidence': 0.9995282888412476}],
  'book': [{'text': 'Brandon Sanderson series',
    'confidence': 0.7838425636291504}]}}

In [47]:
query = "Brandon Sanderson series with a girl with long hair that takes place in Space"

schema = {
    "author": "The exact first and last name of the author.",
    "exact_title": "The official published proper-noun title of the novel.",
    "visual_description": "Physical descriptions of the cover, characters, or aesthetic.",
    "setting": "The location, time period, or world where the story takes place."
}

result = extractor.extract_entities(
    query,
    schema,
    include_confidence=True
)
result

{'entities': {'author': [{'text': 'Brandon Sanderson',
    'confidence': 0.9989689588546753}],
  'exact_title': [],
  'visual_description': [{'text': 'girl with long hair',
    'confidence': 0.6109878420829773}],
  'setting': [{'text': 'Space', 'confidence': 0.998908281326294}]}}

In [48]:
query = "book where there was a sword on the cover, but it looked kind of old and had oldish lettering. It had characters with powers too"

result = extractor.extract_entities(query, ["author", "book"],
    include_confidence=True)
result

{'entities': {'author': [],
  'book': [{'text': 'book', 'confidence': 0.9614467024803162}]}}

In [57]:
query = "book where there was a sword on the cover, but it looked kind of old and had oldish lettering. It had characters with powers too"

schema = {
    "author": "The exact first and last name of the author.",
    "novel_title": "The official published proper-noun title of the novel."
}

result = extractor.extract_entities(
    query,
    schema,
    include_confidence=True
)
result

{'entities': {'author': [],
  'novel_title': [{'text': 'book', 'confidence': 0.9055392742156982}]}}

In [61]:
query = "The one Defensive Baking series, I think by something Kingfisher"

schema = {
    "author": "The exact first and last name of the author.",
    "novel_title": "The official published proper-noun title of the novel."
}

result = extractor.extract_entities(
    query,
    schema,
    include_confidence=True
)
result

{'entities': {'author': [{'text': 'something Kingfisher',
    'confidence': 0.9999998807907104}],
  'novel_title': [{'text': 'one Defensive Baking',
    'confidence': 0.9999722242355347}]}}

In [62]:
query = "The one Defensive Baking series, I think by something Kingfisher"

schema = {
    "author": "The proper noun name of the author. CRITICAL: Do not include conversational filler words like 'something', 'maybe', or 'by' at the beginning of the name.",
    "novel_title": "The proper noun title. CRITICAL: Do not include quantities like 'one' or descriptors like 'series' unless they are part of the official title."
}

result = extractor.extract_entities(
    query,
    schema,
    include_confidence=True
)
result

{'entities': {'author': [{'text': 'Kingfisher',
    'confidence': 0.9831699132919312}],
  'novel_title': [{'text': 'Defensive Baking series',
    'confidence': 0.9196973443031311}]}}

In [63]:
query = "book where there was a sword on the cover, but it looked kind of old and had oldish lettering. It had characters with powers too"

schema = {
    "author": "The proper noun name of the author. CRITICAL: Do not include conversational filler words like 'something', 'maybe', or 'by' at the beginning of the name.",
    "novel_title": "The proper noun title. CRITICAL: Do not include quantities like 'one' or descriptors like 'series' unless they are part of the official title."
}

result = extractor.extract_entities(
    query,
    schema,
    include_confidence=True
)
result

{'entities': {'author': [],
  'novel_title': [{'text': 'book', 'confidence': 0.9591628313064575}]}}

In [64]:
query = "sword on the cover, but it looked kind of old and had oldish lettering. It had characters with powers too"

schema = {
    "author": "The proper noun name of the author. CRITICAL: Do not include conversational filler words like 'something', 'maybe', or 'by' at the beginning of the name.",
    "novel_title": "The proper noun title. CRITICAL: Do not include quantities like 'one' or descriptors like 'series' unless they are part of the official title."
}

result = extractor.extract_entities(
    query,
    schema,
    include_confidence=True
)
result

{'entities': {'author': [],
  'novel_title': [{'text': 'sword', 'confidence': 0.9988611936569214}]}}

In [ ]:
query = "sword on the cover, but it looked kind of old and had oldish lettering. It had characters with powers too"

schema = {
    "author": "The exact first and last name of the author.",
    "novel_title": "The official published proper-noun title of the novel."
}

result = extractor.extract_entities(
    query,
    schema,
    include_confidence=True
)
result

{'entities': {'author': [],
  'novel_title': [{'text': 'sword', 'confidence': 0.9998399019241333}]}}

In [ ]:
query = "sword on the cover, but it looked kind of old and had oldish lettering. It had characters with powers too"

schema = {
    "author": "The exact first and last name of the author.",
    "novel_title": "The official published proper-noun title of the novel.",
    "visual_description": "Physical descriptions of the cover, characters, or aesthetic.",
    "setting": "The location, time period, or world where the story takes place."
}

result = extractor.extract_entities(
    query,
    schema,
    include_confidence=True
)
result

{'entities': {'author': [],
  'novel_title': [{'text': 'sword', 'confidence': 0.969730794429779}],
  'visual_description': [{'text': 'oldish lettering',
    'confidence': 0.7185609936714172},
   {'text': 'characters', 'confidence': 0.5966821908950806}],
  'setting': []}}